<a href="https://colab.research.google.com/github/alfredqbit/NU-DDS-8536/blob/main/GR_CS_HJEPA_Chapter4_Phase1_Review_Freeze_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GR-CS-HJEPA Chapter 4: Phase 1 Review, Production Hardening, and Confirmatory Protocol Freeze

This notebook is the **next step after successful Phase 1 pilot runs**. It does three things:

1. Adds production-review code that checks whether Phase 1 metrics are scientifically meaningful, not just whether they exist.
2. Runs strict quality gates for noncollapse, downstream-head usefulness, and routing sanity.
3. Freezes a **conditional confirmatory protocol** with fixed seed lists, endpoints, failure rules, and launch gates.

Expected outcome from the uploaded Phase 1 run: the infrastructure should pass, but the confirmatory launch should remain on hold until Phase 1B fixes noncollapse and downstream-head readiness.

In [ ]:
# === 1. Runtime setup ===
from pathlib import Path
import os, json, textwrap, shutil, subprocess, sys

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

MOUNT_DRIVE = True
REPO_URL = "https://github.com/alfredqbit/grcshjepa.git"  # edit if needed
PROJECT = "grcshjepa"
PROJECT_ROOT = Path(f"/content/{PROJECT}") if IN_COLAB else Path.cwd()

if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')

if IN_COLAB:
    os.chdir('/content')
    if PROJECT_ROOT.exists():
        os.chdir(PROJECT_ROOT)
        # Use pull only when this is a real git repo.
        if (PROJECT_ROOT / '.git').exists():
            subprocess.run(['git', 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=False)
        if not PROJECT_ROOT.exists():
            PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_ROOT)

print('PROJECT_ROOT =', Path.cwd())

In [ ]:
# === 2. Ensure package folders exist ===
for d in [
    'src/grcshjepa/production', 'tests', 'scripts', 'configs',
    'analysis/phase1_review', 'analysis/confirmatory_protocol_v1'
]:
    Path(d).mkdir(parents=True, exist_ok=True)

Path('src/grcshjepa/__init__.py').write_text('"""GR-CS-HJEPA Chapter 4 experimental package."""\n\n__version__ = "0.2.0-phase1-review"\n')
Path('src/grcshjepa/production/__init__.py').write_text('"""Production review and confirmatory protocol utilities."""\n')
Path('.gitignore').write_text('__pycache__/\n*.py[cod]\n*.egg-info/\n.pytest_cache/\n.ipynb_checkpoints/\nruns/\nanalysis/*/large_artifacts/\n*.pt\n*.pth\n*.tar.gz\n*.zip\n.DS_Store\n')
print('Folders and package init files ready.')

In [ ]:
# === 3. Write strict Phase 1 quality-gate module ===
Path('src/grcshjepa/production/quality_gates.py').write_text('from __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, asdict\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport yaml\n\n\n@dataclass(frozen=True)\nclass GateResult:\n    gate: str\n    passed: bool\n    severity: str\n    value: Any\n    threshold: Any\n    interpretation: str\n    action: str\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef load_phase1_results(results_dir: str | Path) -> pd.DataFrame:\n    root = Path(results_dir)\n    preferred = root / "phase1_pilot_combined_results.csv"\n    if preferred.exists():\n        return pd.read_csv(preferred)\n    frames: list[pd.DataFrame] = []\n    for p in sorted(root.glob("**/*.csv")):\n        if "combined" in p.name or "summary" in p.name or "readiness" in p.name:\n            continue\n        try:\n            df = pd.read_csv(p)\n            df["source_csv"] = str(p)\n            frames.append(df)\n        except Exception:\n            continue\n    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()\n\n\ndef _mean(df: pd.DataFrame, column: str) -> float:\n    vals = pd.to_numeric(df.get(column, pd.Series(dtype=float)), errors="coerce").dropna()\n    return float(vals.mean()) if len(vals) else float("nan")\n\n\ndef _min(df: pd.DataFrame, column: str) -> float:\n    vals = pd.to_numeric(df.get(column, pd.Series(dtype=float)), errors="coerce").dropna()\n    return float(vals.min()) if len(vals) else float("nan")\n\n\ndef evaluate_phase1_quality_gates(\n    df: pd.DataFrame,\n    *,\n    min_effective_rank: float = 6.0,\n    min_cov_trace: float = 1.0,\n    max_ac_loss: float = 8.0,\n    maze_random_acc: float = 0.25,\n    min_maze_acc_advantage: float = 0.10,\n    max_sort_mse: float = 0.12,\n    min_sort_exactish: float = 0.05,\n) -> list[GateResult]:\n    """Evaluate whether Phase 1 outputs are meaningful enough to launch confirmation.\n\n    The gates are intentionally stricter than the original readiness report.  The original report\n    checked that metrics existed.  These gates check whether those metrics show a representation\n    and downstream readout worth confirming.\n    """\n    gates: list[GateResult] = []\n    if df.empty:\n        return [GateResult("infrastructure_rows", False, "blocker", 0, ">0", "No Phase 1 rows were loaded.", "Rerun Phase 1.")]\n\n    failure_rate = float((df.get("status", pd.Series(dtype=str)).fillna("") == "failure").mean())\n    gates.append(GateResult(\n        "infrastructure_failure_rate",\n        failure_rate <= 0.10,\n        "blocker" if failure_rate > 0.10 else "pass",\n        failure_rate,\n        "<=0.10",\n        "All or nearly all pilot jobs must complete before metric interpretation is useful.",\n        "Fix failing jobs or classify administrative failures under the same seeds.",\n    ))\n\n    studies = sorted(str(x) for x in df.get("study", pd.Series(dtype=str)).dropna().unique())\n    gates.append(GateResult(\n        "infrastructure_studies_present",\n        set(["study1", "study2", "study3"]).issubset(studies),\n        "blocker" if not set(["study1", "study2", "study3"]).issubset(studies) else "pass",\n        studies,\n        "study1, study2, study3",\n        "The pilot must cover pretraining, downstream heads, and routing metrics.",\n        "Rerun missing studies before freezing.",\n    ))\n\n    s1 = df[df.get("study") == "study1"] if "study" in df else pd.DataFrame()\n    min_er = _min(s1, "val_effective_rank")\n    gates.append(GateResult(\n        "study1_noncollapse_effective_rank",\n        np.isfinite(min_er) and min_er >= min_effective_rank,\n        "blocker" if not (np.isfinite(min_er) and min_er >= min_effective_rank) else "pass",\n        round(min_er, 4) if np.isfinite(min_er) else None,\n        f">={min_effective_rank}",\n        "Low prediction loss is not meaningful if the latent covariance is low-rank.",\n        "Increase anti-collapse strength, training horizon, and projector dimensionality; rerun Phase 1B.",\n    ))\n\n    min_cov = _min(s1, "val_cov_trace")\n    gates.append(GateResult(\n        "study1_noncollapse_cov_trace",\n        np.isfinite(min_cov) and min_cov >= min_cov_trace,\n        "blocker" if not (np.isfinite(min_cov) and min_cov >= min_cov_trace) else "pass",\n        round(min_cov, 6) if np.isfinite(min_cov) else None,\n        f">={min_cov_trace}",\n        "A near-zero covariance trace indicates that the target/online latent branches may have collapsed together.",\n        "Use the strengthened anti-collapse loss and treat prediction loss as invalid until this passes.",\n    ))\n\n    ac_loss = _mean(s1, "train_anti_collapse")\n    gates.append(GateResult(\n        "study1_anti_collapse_loss_resolved",\n        np.isfinite(ac_loss) and ac_loss <= max_ac_loss,\n        "blocker" if not (np.isfinite(ac_loss) and ac_loss <= max_ac_loss) else "pass",\n        round(ac_loss, 4) if np.isfinite(ac_loss) else None,\n        f"<={max_ac_loss}",\n        "The observed value should move far below the projection dimension-scale collapsed penalty.",\n        "Do not interpret latent-prediction loss as representation learning until this improves.",\n    ))\n\n    s2 = df[df.get("study") == "study2"] if "study" in df else pd.DataFrame()\n    maze = s2[s2.get("task") == "maze_action"] if "task" in s2 else pd.DataFrame()\n    maze_acc = _mean(maze, "test_acc")\n    maze_thresh = maze_random_acc + min_maze_acc_advantage\n    gates.append(GateResult(\n        "study2_maze_head_above_random",\n        np.isfinite(maze_acc) and maze_acc >= maze_thresh,\n        "blocker" if not (np.isfinite(maze_acc) and maze_acc >= maze_thresh) else "pass",\n        round(maze_acc, 4) if np.isfinite(maze_acc) else None,\n        f">={maze_thresh:.2f}",\n        "A four-action maze head near 0.25 is indistinguishable from random readout.",\n        "Rerun after noncollapse is fixed and use verifier-derived action labels.",\n    ))\n\n    sort = s2[s2.get("task") == "sorting_head"] if "task" in s2 else pd.DataFrame()\n    sort_mse = _mean(sort, "test_mse")\n    sort_exact = _mean(sort, "exactish_rate")\n    gates.append(GateResult(\n        "study2_sorting_head_learns",\n        (np.isfinite(sort_mse) and sort_mse <= max_sort_mse) or (np.isfinite(sort_exact) and sort_exact >= min_sort_exactish),\n        "blocker" if not ((np.isfinite(sort_mse) and sort_mse <= max_sort_mse) or (np.isfinite(sort_exact) and sort_exact >= min_sort_exactish)) else "pass",\n        {"test_mse": round(sort_mse, 4) if np.isfinite(sort_mse) else None, "exactish_rate": round(sort_exact, 4) if np.isfinite(sort_exact) else None},\n        {"test_mse": f"<={max_sort_mse}", "exactish_rate": f">={min_sort_exactish}"},\n        "A downstream regression head with zero exact-ish success is not evidence that the pretrained representation is useful.",\n        "Use stronger pretraining and compare against a naive sorted-sequence baseline.",\n    ))\n\n    s3 = df[df.get("study") == "study3"] if "study" in df else pd.DataFrame()\n    finite_surface = bool(len(s3) and np.isfinite(pd.to_numeric(s3.get("normalized_surface"), errors="coerce")).all())\n    traffic_degrad = pd.to_numeric(s3.get("traffic_degradation", pd.Series(dtype=float)), errors="coerce").dropna()\n    traffic_ok = bool(len(traffic_degrad) and (traffic_degrad >= -1e-8).all())\n    variants = sorted(str(x) for x in s3.get("variant", pd.Series(dtype=str)).dropna().unique())\n    gates.append(GateResult(\n        "study3_routing_metric_sanity",\n        finite_surface and traffic_ok and {"sparsity", "euclidean_length", "tube_only", "full_surface"}.issubset(variants),\n        "pass" if (finite_surface and traffic_ok) else "review",\n        {"finite_surface": finite_surface, "traffic_nonnegative": traffic_ok, "variants": variants},\n        "finite surfaces; nonnegative traffic degradation; all variants present",\n        "Routing metrics are meaningful as an engineering sanity check, but still toy unless coupled to learned task performance.",\n        "Replace toy routing with learned route gates before confirmatory Study 3.",\n    ))\n    return gates\n\n\ndef gates_to_frame(gates: list[GateResult]) -> pd.DataFrame:\n    return pd.DataFrame([g.to_dict() for g in gates])\n\n\ndef pilot_metric_judgment(df: pd.DataFrame) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    s1 = df[df.get("study") == "study1"] if "study" in df else pd.DataFrame()\n    rows.append({\n        "area": "Study 1 latent prediction",\n        "judgment": "not meaningful as evidence of representation quality",\n        "reason": "validation prediction losses are tiny, but effective rank and covariance trace are too low and anti-collapse loss remains at the collapsed dimension-scale penalty",\n        "action": "treat prediction loss as a smoke metric only; rerun Phase 1B with stronger anti-collapse and noncollapse launch gates",\n        "key_values": {\n            "min_val_effective_rank": _min(s1, "val_effective_rank"),\n            "min_val_cov_trace": _min(s1, "val_cov_trace"),\n            "mean_train_anti_collapse": _mean(s1, "train_anti_collapse"),\n        },\n    })\n    s2 = df[df.get("study") == "study2"] if "study" in df else pd.DataFrame()\n    rows.append({\n        "area": "Study 2 frozen downstream heads",\n        "judgment": "meaningful as a negative readiness signal, not as support",\n        "reason": "maze action accuracy is near random four-class performance and sorting exactish success is zero in the pilot",\n        "action": "do not launch confirmatory transfer tests until heads beat naive/random baselines after representation noncollapse is fixed",\n        "key_values": {\n            "maze_action_test_acc_mean": _mean(s2[s2.get("task") == "maze_action"] if "task" in s2 else pd.DataFrame(), "test_acc"),\n            "sorting_test_mse_mean": _mean(s2[s2.get("task") == "sorting_head"] if "task" in s2 else pd.DataFrame(), "test_mse"),\n            "sorting_exactish_mean": _mean(s2[s2.get("task") == "sorting_head"] if "task" in s2 else pd.DataFrame(), "exactish_rate"),\n        },\n    })\n    s3 = df[df.get("study") == "study3"] if "study" in df else pd.DataFrame()\n    base = s3[s3.get("damage_type") == "none"] if "damage_type" in s3 else pd.DataFrame()\n    by_variant = base.groupby("variant")["normalized_surface"].mean().to_dict() if "variant" in base else {}\n    rows.append({\n        "area": "Study 3 routing metrics",\n        "judgment": "meaningful as routing-code sanity, not dissertation evidence yet",\n        "reason": "surface, traffic, damage and variant metrics are finite and directionally interpretable, but the current graph is a synthetic stand-in not learned from the H-JEPA model",\n        "action": "freeze the metric definitions; replace toy routing with learned route gates before confirmatory Study 3",\n        "key_values": by_variant,\n    })\n    return pd.DataFrame(rows)\n\n\ndef write_quality_outputs(df: pd.DataFrame, output_dir: str | Path) -> dict[str, Path]:\n    out = Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    gates = evaluate_phase1_quality_gates(df)\n    gate_df = gates_to_frame(gates)\n    judgment_df = pilot_metric_judgment(df)\n    gate_df.to_csv(out / "phase1_quality_gates.csv", index=False)\n    judgment_df.to_csv(out / "pilot_metric_judgment.csv", index=False)\n    (out / "phase1_quality_gates.json").write_text(json.dumps([g.to_dict() for g in gates], indent=2, default=str))\n    return {\n        "quality_gates_csv": out / "phase1_quality_gates.csv",\n        "quality_gates_json": out / "phase1_quality_gates.json",\n        "pilot_metric_judgment_csv": out / "pilot_metric_judgment.csv",\n    }\n\n\ndef all_launch_gates_pass(gates: list[GateResult]) -> bool:\n    return all(g.passed for g in gates if g.severity == "blocker")\n')
print('Wrote src/grcshjepa/production/quality_gates.py')

In [ ]:
# === 4. Write confirmatory protocol freeze module ===
Path('src/grcshjepa/production/protocol.py').write_text('from __future__ import annotations\n\nimport json\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\nimport yaml\n\nfrom grcshjepa.production.quality_gates import (\n    all_launch_gates_pass,\n    evaluate_phase1_quality_gates,\n    gates_to_frame,\n    load_phase1_results,\n    pilot_metric_judgment,\n    write_quality_outputs,\n)\nfrom grcshjepa.utils import git_commit_hash, save_json\n\n\ndef default_confirmatory_protocol(launchable: bool) -> dict[str, Any]:\n    """Return the frozen confirmatory protocol skeleton.\n\n    The protocol may be frozen even when it is not launchable.  In that case the launch status is\n    conditional-hold, and a Phase 1B production-hardening rerun must satisfy all blocker gates.\n    """\n    return {\n        "protocol_id": "GR-CS-HJEPA-CONFIRMATORY-V1",\n        "frozen_at_utc": datetime.now(timezone.utc).isoformat(),\n        "git_commit_at_freeze": git_commit_hash(),\n        "launch_status": "launchable" if launchable else "conditional_hold_not_launchable",\n        "reason_if_hold": None if launchable else "Phase 1 pilot infrastructure passed, but noncollapse and downstream-head readiness gates failed. Confirmatory seeds must not start until Phase 1B passes all blocker gates.",\n        "global_rules": {\n            "primary_experimental_unit": "independently initialized and trained model seed",\n            "confirmatory_seed_policy": "seeds are fixed before launch and are not replaced except for documented administrative hardware failure rerun under the same seed",\n            "pilot_exclusion_rule": "Phase 0 and Phase 1 pilot seeds are excluded from confirmatory inference",\n            "failure_rule": "model-dependent failures remain in the primary analysis using the prespecified failure score or joint success/failure model",\n            "multiplicity": "Holm control over the primary hypothesis family; FDR for secondary diagnostics",\n            "analysis_lock": "analysis scripts are run once on blinded labels before unblinding primary variant labels",\n        },\n        "launch_gates": {\n            "G0_tests": "all unit tests and numerical sanity tests pass on a clean runtime",\n            "G1_noncollapse": "every primary H-JEPA variant has mean validation effective rank >= 6.0, covariance trace >= 1.0, and anti-collapse loss <= 8.0 on Phase 1B",\n            "G2_downstream_heads": "maze action head exceeds 0.35 accuracy and sorting head achieves test MSE <= 0.12 or exactish_rate >= 0.05 using frozen backbone representations",\n            "G3_routing": "routing metrics are finite, damage reduces delivered traffic, and production Study 3 uses learned route gates coupled to model training rather than the toy graph generator",\n            "G4_protocol": "configs, seed lists, hyperparameter budgets, endpoints, failure rules, and analysis scripts have frozen hashes",\n        },\n        "confirmatory_studies": {\n            "study1_predictive_pretraining": {\n                "status": "frozen_conditional",\n                "seed_list": list(range(1000, 1016)),\n                "primary_variants": ["GR-CS-HJEPA-spiking", "flat-HJEPA-MLP"],\n                "secondary_variants": ["nonspiking-HJEPA", "supervised-only-GR-CS-HJEPA"],\n                "primary_endpoint": "held-out normalized latent prediction error subject to noncollapse validity gates",\n                "primary_contrast": "GR-CS-HJEPA-spiking minus flat-HJEPA-MLP",\n                "margin": 0.02,\n                "decision_rule": "noninferior if the adjusted upper confidence bound on excess normalized prediction loss is below 0.02 and all noncollapse gates pass",\n            },\n            "study2_low_shot_downstream_heads": {\n                "status": "frozen_conditional",\n                "seed_list": list(range(1100, 1116)),\n                "primary_variants": ["pretrained-GR-CS-HJEPA-frozen-backbone", "random-frozen-backbone", "supervised-head-only-control"],\n                "primary_endpoints": ["maze valid-action accuracy", "sorting exactish_rate and MSE"],\n                "primary_contrast": "pretrained frozen backbone versus strongest prespecified control",\n                "decision_rule": "supported only if pretrained representation improves the primary endpoint and confidence interval excludes the smallest useful improvement threshold",\n            },\n            "study3_surface_and_damage": {\n                "status": "frozen_conditional",\n                "seed_list": list(range(2000, 2020)),\n                "primary_variants": ["full_surface", "euclidean_length"],\n                "secondary_variants": ["sparsity", "tube_only", "no_geometry"],\n                "primary_surface_endpoint": "log normalized surface ratio log(S_full/S_euclidean)",\n                "surface_margin": "log(0.85)",\n                "damage_endpoint": "degradation in task-valid solution rate under uniform and spatial routing damage at 5%, 10%, and 15%",\n                "decision_rule": "surface hypothesis supported only if full_surface reduces normalized surface by at least 15% and task prediction remains noninferior; damage robustness requires at least five percentage points smaller degradation",\n            },\n        },\n        "files_to_hash_before_launch": [\n            "configs/confirmatory_protocol_v1.yaml",\n            "configs/study1_confirmatory.yaml",\n            "configs/study2_confirmatory.yaml",\n            "configs/study3_confirmatory.yaml",\n            "src/grcshjepa/",\n            "scripts/run_confirmatory_study1.py",\n            "scripts/run_confirmatory_study2.py",\n            "scripts/run_confirmatory_study3.py",\n            "scripts/analyze_confirmatory.py",\n        ],\n    }\n\n\ndef write_confirmatory_protocol(protocol: dict[str, Any], output_dir: str | Path) -> dict[str, Path]:\n    out = Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    yaml_path = out / "confirmatory_protocol_v1.yaml"\n    json_path = out / "confirmatory_protocol_v1.json"\n    md_path = out / "CONFIRMATORY_PROTOCOL.md"\n    yaml_path.write_text(yaml.safe_dump(protocol, sort_keys=False))\n    json_path.write_text(json.dumps(protocol, indent=2, default=str))\n    md_path.write_text(protocol_to_markdown(protocol))\n    return {"yaml": yaml_path, "json": json_path, "markdown": md_path}\n\n\ndef protocol_to_markdown(protocol: dict[str, Any]) -> str:\n    lines = [\n        "# Frozen Confirmatory Protocol V1",\n        "",\n        f"Protocol ID: `{protocol[\'protocol_id\']}`",\n        f"Frozen at UTC: `{protocol[\'frozen_at_utc\']}`",\n        f"Launch status: **{protocol[\'launch_status\']}**",\n    ]\n    if protocol.get("reason_if_hold"):\n        lines += ["", f"**Reason for hold:** {protocol[\'reason_if_hold\']}"]\n    lines += ["", "## Global Rules", ""]\n    for k, v in protocol["global_rules"].items():\n        lines.append(f"- **{k}:** {v}")\n    lines += ["", "## Launch Gates", ""]\n    for k, v in protocol["launch_gates"].items():\n        lines.append(f"- **{k}:** {v}")\n    lines += ["", "## Confirmatory Studies", ""]\n    for name, spec in protocol["confirmatory_studies"].items():\n        lines += [f"### {name}", ""]\n        for k, v in spec.items():\n            lines.append(f"- **{k}:** {v}")\n        lines.append("")\n    return "\\n".join(lines)\n\n\ndef write_decision_memo(df: pd.DataFrame, gates: list, judgment: pd.DataFrame, protocol: dict[str, Any], output_path: str | Path) -> None:\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    gate_df = gates_to_frame(gates)\n    blockers = gate_df[(gate_df["severity"] == "blocker") & (~gate_df["passed"])]\n    lines = [\n        "# Phase 1 Review, Production Fixes, and Confirmatory Freeze Decision",\n        "",\n        "## Decision",\n        "",\n        f"Confirmatory launch status: **{protocol[\'launch_status\']}**.",\n    ]\n    if len(blockers):\n        lines.append("The protocol is frozen as a conditional document, but confirmatory runs should not begin. Phase 1 showed working infrastructure but failed representation and downstream-readout readiness gates.")\n    else:\n        lines.append("All blocker gates passed. Confirmatory runs may begin under the frozen protocol.")\n    lines += ["", "## Pilot Metric Judgment", ""]\n    for _, row in judgment.iterrows():\n        lines += [f"### {row[\'area\']}", f"- Judgment: {row[\'judgment\']}", f"- Reason: {row[\'reason\']}", f"- Action: {row[\'action\']}", f"- Key values: `{row[\'key_values\']}`", ""]\n    lines += ["", "## Quality Gates", ""]\n    for _, row in gate_df.iterrows():\n        status = "PASS" if row["passed"] else "FAIL"\n        lines.append(f"- **{row[\'gate\']}**: {status}; value={row[\'value\']}; threshold={row[\'threshold\']}; action={row[\'action\']}")\n    lines += ["", "## Production-Code Weaknesses Fixed or Flagged", "",\n              "1. Added strict quality gates; metrics are no longer judged meaningful merely because they exist.",\n              "2. Strengthened anti-collapse diagnostics and loss with variance-hinge terms while retaining the covariance-identity target.",\n              "3. Added a conditional confirmatory protocol with fixed seed lists, endpoints, failure rules, and launch gates.",\n              "4. Added a Phase 1B remediation config for noncollapse/downstream-readout hardening.",\n              "5. Added .gitignore entries to keep pycache, egg-info, runs, archives, and checkpoints out of Git commits.",\n              "", "## Required Phase 1B Before Confirmation", "",\n              "Run the Phase 1B remediation configuration after committing the code fixes. Confirmation is permitted only if noncollapse, downstream-head, and routing-coupling gates pass."\n             ]\n    output_path.write_text("\\n".join(lines))\n\n\ndef review_and_freeze(results_dir: str | Path, output_dir: str | Path, protocol_dir: str | Path) -> dict[str, Any]:\n    df = load_phase1_results(results_dir)\n    qpaths = write_quality_outputs(df, output_dir)\n    gates = evaluate_phase1_quality_gates(df)\n    judgment = pilot_metric_judgment(df)\n    launchable = all_launch_gates_pass(gates)\n    protocol = default_confirmatory_protocol(launchable)\n    ppaths = write_confirmatory_protocol(protocol, protocol_dir)\n    write_decision_memo(df, gates, judgment, protocol, Path(output_dir) / "phase1_decision_memo.md")\n    save_json({\n        "launchable": launchable,\n        "quality_outputs": {k: str(v) for k, v in qpaths.items()},\n        "protocol_outputs": {k: str(v) for k, v in ppaths.items()},\n    }, Path(output_dir) / "phase1_review_manifest.json")\n    return {"launchable": launchable, "protocol": protocol, "quality_paths": qpaths, "protocol_paths": ppaths}\n')
print('Wrote src/grcshjepa/production/protocol.py')

In [ ]:
# === 5. Write scripts, tests, and Phase 1B remediation config ===
Path('scripts/review_phase1_and_freeze_protocol.py').write_text('from __future__ import annotations\n\nimport argparse\nimport pandas as pd\n\nfrom grcshjepa.production.protocol import review_and_freeze\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Review Phase 1 pilot outputs and freeze a conditional confirmatory protocol.")\n    parser.add_argument("--results-dir", default="runs/phase1_pilot")\n    parser.add_argument("--output-dir", default="analysis/phase1_review")\n    parser.add_argument("--protocol-dir", default="analysis/confirmatory_protocol_v1")\n    args = parser.parse_args()\n    result = review_and_freeze(args.results_dir, args.output_dir, args.protocol_dir)\n    print(pd.Series({\n        "launchable": result["launchable"],\n        "protocol_status": result["protocol"]["launch_status"],\n        "decision_memo": f"{args.output_dir}/phase1_decision_memo.md",\n        "protocol_markdown": f"{args.protocol_dir}/CONFIRMATORY_PROTOCOL.md",\n    }).to_string())\n\n\nif __name__ == "__main__":\n    main()\n')
Path('tests/test_production_review.py').write_text('from pathlib import Path\n\nimport pandas as pd\n\nfrom grcshjepa.production.quality_gates import evaluate_phase1_quality_gates, all_launch_gates_pass\nfrom grcshjepa.production.protocol import default_confirmatory_protocol\n\n\ndef test_quality_gates_catch_collapsed_representation():\n    df = pd.DataFrame([\n        {"study": "study1", "status": "complete", "seed": 0, "val_effective_rank": 1.2, "val_cov_trace": 0.001, "train_anti_collapse": 16.0},\n        {"study": "study2", "status": "complete", "seed": 0, "task": "maze_action", "test_acc": 0.25},\n        {"study": "study2", "status": "complete", "seed": 0, "task": "sorting_head", "test_mse": 0.25, "exactish_rate": 0.0},\n        {"study": "study3", "status": "complete", "seed": 0, "variant": "full_surface", "damage_type": "none", "normalized_surface": 0.1},\n    ])\n    gates = evaluate_phase1_quality_gates(df)\n    assert not all_launch_gates_pass(gates)\n    assert any(g.gate == "study1_noncollapse_effective_rank" and not g.passed for g in gates)\n\n\ndef test_conditional_protocol_has_fixed_seed_lists():\n    protocol = default_confirmatory_protocol(False)\n    assert protocol["launch_status"] == "conditional_hold_not_launchable"\n    assert protocol["confirmatory_studies"]["study1_predictive_pretraining"]["seed_list"][0] == 1000\n    assert len(protocol["confirmatory_studies"]["study3_surface_and_damage"]["seed_list"]) == 20\n')
Path('configs/phase1b_remediation.yaml').write_text('# Phase 1B remediation: stronger noncollapse and downstream-readout pilot.\n# Do not use these runs as confirmatory results.\nproject: GR-CS-HJEPA-Chapter4\nsmoke_mode: false\nseed: 0\ndevice: auto\noutput_dir: runs/phase1b_remediation\nmaze_size_train: 12\nmaze_size_ood: 16\nmaze_obstacle_prob: 0.22\nsorting_length_train: 12\nsorting_length_ood: 16\ntrain_samples: 1024\nval_samples: 256\ntest_samples: 256\nlatent_dim: 64\nprojection_dim: 32\nhorizon_set: [1, 2, 3, 4]\nbatch_size: 64\nepochs: 12\nlr: 0.001\nema_tau: 0.995\nlambda_ac: 0.20\nlambda_suff: 0.0\nlambda_alias: 0.0\nlambda_unc: 0.0\nn_neurons: 96\nn_filters: 3\ninternal_steps: 8\ndt: 0.06\nnoise_std: 0.005\nsurrogate_eps: 0.7\ndownstream_label_fraction: 0.10\nhead_epochs: 25\nhead_lr: 0.001\nrouting_nodes: 128\nrouting_segments: 384\nrouting_terminals: 20\ndamage_levels: [0.05, 0.10, 0.15]\n')

for name in ['study1_confirmatory.yaml', 'study2_confirmatory.yaml', 'study3_confirmatory.yaml']:
    p = Path('configs') / name
    if not p.exists():
        p.write_text('# Frozen placeholder generated by protocol freeze. Fill only through approved protocol revision.
protocol: GR-CS-HJEPA-CONFIRMATORY-V1
config_status: conditional_hold_not_launchable
')

print('Wrote scripts, tests, and configs.')

In [ ]:
# === 6. Install editable package and run tests ===
# If build isolation fails in Colab, PYTHONPATH fallback still lets the notebook run.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=False)
os.environ['PYTHONPATH'] = str(Path('src').resolve()) + os.pathsep + os.environ.get('PYTHONPATH', '')

result = subprocess.run(['pytest', '-q'], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Tests failed. Fix tests before reviewing Phase 1 metrics.')

In [ ]:
# === 7. Locate Phase 1 pilot results ===
from pathlib import Path
candidates = [
    Path('runs/phase1_pilot'),
    Path('/content/drive/MyDrive/grcshjepa_artifacts/phase1_pilot') if IN_COLAB else Path('___not_colab___'),
]
for c in candidates:
    print(c, 'exists=', c.exists())

RESULTS_DIR = next((c for c in candidates if (c / 'phase1_pilot_combined_results.csv').exists()), None)
if RESULTS_DIR is None:
    raise FileNotFoundError('Could not find runs/phase1_pilot/phase1_pilot_combined_results.csv. Run Phase 1 pilot notebook first or copy results into runs/phase1_pilot.')
print('Using RESULTS_DIR =', RESULTS_DIR)

In [ ]:
# === 8. Review Phase 1 metrics and freeze conditional confirmatory protocol ===
cmd = [
    sys.executable, 'scripts/review_phase1_and_freeze_protocol.py',
    '--results-dir', str(RESULTS_DIR),
    '--output-dir', 'analysis/phase1_review',
    '--protocol-dir', 'analysis/confirmatory_protocol_v1',
]
res = subprocess.run(cmd, text=True, capture_output=True)
print(res.stdout)
print(res.stderr)
if res.returncode != 0:
    raise RuntimeError('Phase 1 review/protocol freeze script failed.')

# Copy frozen protocol YAML into configs for repository tracking.
shutil.copyfile('analysis/confirmatory_protocol_v1/confirmatory_protocol_v1.yaml', 'configs/confirmatory_protocol_v1.yaml')
print('Copied confirmatory protocol to configs/confirmatory_protocol_v1.yaml')

In [ ]:
# === 9. Display quality gates and pilot metric judgment ===
import pandas as pd
from IPython.display import display, Markdown

gates = pd.read_csv('analysis/phase1_review/phase1_quality_gates.csv')
judgment = pd.read_csv('analysis/phase1_review/pilot_metric_judgment.csv')

display(Markdown('## Quality Gates'))
display(gates)

display(Markdown('## Pilot Metric Judgment'))
display(judgment)

display(Markdown(Path('analysis/phase1_review/phase1_decision_memo.md').read_text()))

In [ ]:
# === 10. Show frozen protocol summary ===
import yaml
protocol = yaml.safe_load(Path('analysis/confirmatory_protocol_v1/confirmatory_protocol_v1.yaml').read_text())
print('Protocol ID:', protocol['protocol_id'])
print('Launch status:', protocol['launch_status'])
print('Reason if hold:', protocol.get('reason_if_hold'))
for study, spec in protocol['confirmatory_studies'].items():
    print('
', study)
    print('  seeds:', spec.get('seed_list')[:3], '...', spec.get('seed_list')[-3:], 'n=', len(spec.get('seed_list')))
    print('  endpoint:', spec.get('primary_endpoint') or spec.get('primary_endpoints') or spec.get('primary_surface_endpoint'))

In [ ]:
# === 11. Archive review outputs to Drive if available ===
archive_name = 'grcshjepa_phase1_review_freeze_outputs.tar.gz'
subprocess.run(['tar', '-czf', archive_name, 'analysis/phase1_review', 'analysis/confirmatory_protocol_v1', 'configs/confirmatory_protocol_v1.yaml', 'configs/phase1b_remediation.yaml'], check=True)
print('Wrote', archive_name)

if IN_COLAB and Path('/content/drive/MyDrive').exists():
    dest = Path('/content/drive/MyDrive/grcshjepa_artifacts')
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(archive_name, dest / archive_name)
    print('Copied archive to', dest / archive_name)
else:
    print('Drive not mounted or unavailable; archive remains in project root.')

## Interpretation

A **conditional hold** is the expected honest outcome if Phase 1 infrastructure succeeded but representation quality did not. Do not start confirmatory seeds until Phase 1B passes the noncollapse, downstream-head, and routing-coupling launch gates.

The frozen protocol is still useful: it prevents moving endpoints, seeds, or thresholds after seeing more results. The protocol freezes what will count as success, while the hold prevents weak pilot metrics from being treated as dissertation evidence.